# Blob Correction Table and Concentration Conditions

**Proposition 8 verification.** The Cauchy blob regularisation
$h_\varepsilon(d) = -\tfrac12\ln(d^2+2\varepsilon^2)$ introduces an $O(\varepsilon^2)$
correction to every Havelock eigenvalue:

$$
\lambda_m(\varepsilon) = \lambda_m(0) + \left[-\frac{N^2-1}{12} + P_m\right]\varepsilon^2 + O(\varepsilon^4)
$$

where $P_m$ is the projection of the $O(\varepsilon^2)$ Hessian correction onto the
critical radial Fourier mode $m = \lfloor N/2 \rfloor$.

This notebook:
1. Computes $P_m$ for $N=3,\dots,8$ and verifies against exact values.
2. Builds the full correction table $c_m = -(N^2-1)/12 + P_m$.
3. Analyses stability consequences (sign change at $N=6$).
4. Computes the concentration condition $\delta(N)$.
5. Verifies the operator norm bound numerically.

## Setup

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
from fractions import Fraction

from planetary_polygons.extensions.blob_correction import (
    compute_Pm,
    blob_correction_table,
    stabilization_threshold,
    multipole_far_field_residual,
)

## P_m projection values

The projection $P_m$ of the blob Hessian correction onto the critical radial
Fourier mode at wavenumber $m = \lfloor N/2\rfloor$.

Paper exact values (table in section 6.4):

| N | P_m (exact) |
|---|-------------|
| 3 | -1/3 |
| 4 | 1/4 |
| 5 | 1 |
| 6 | 4 |
| 7 | 8 |
| 8 | 65/4 |

In [ ]:
EXACT_PM = {
    3: Fraction(-1, 3),
    4: Fraction(1, 4),
    5: Fraction(1, 1),
    6: Fraction(4, 1),
    7: Fraction(8, 1),
    8: Fraction(65, 4),
}

print(f"{'N':>3}  {'P_m (computed)':>14}  {'P_m (exact)':>12}  {'rel error':>10}")
print("-" * 50)

for N in range(3, 9):
    pm_num = compute_Pm(N)
    pm_exact = float(EXACT_PM[N])
    rel_err = abs(pm_num - pm_exact) / (abs(pm_exact) + 1e-15)
    status = "OK" if rel_err < 0.05 else "FAIL"
    print(f"{N:>3}  {pm_num:>14.6f}  {str(EXACT_PM[N]):>12s}  {rel_err:>10.2e}  {status}")
    assert rel_err < 0.05, f"P_m mismatch at N={N}"

## Total correction c_m = -(N^2 - 1)/12 + P_m

The total $O(\varepsilon^2)$ eigenvalue correction is

$$
c_m = -\frac{N^2-1}{12} + P_m
$$

The sign of $c_m$ determines whether the blob regularisation stabilises
or destabilises the polygon. The universal shift $-(N^2-1)/12$ is always
negative; for large enough $N$ the mode-dependent $P_m$ dominates.

In [ ]:
EXPECTED_CM = {
    3: Fraction(-1, 1),
    4: Fraction(-1, 1),
    5: Fraction(-1, 1),
    6: Fraction(13, 12),
    7: Fraction(4, 1),
    8: Fraction(11, 1),
}

table = blob_correction_table()

print(f"{'N':>3}  {'-(N^2-1)/12':>12}  {'P_m':>10}  {'c_m':>10}  {'c_m (exact)':>12}  {'sign':>6}")
print("=" * 65)

for N in range(3, 9):
    row = table[N]
    shift = row['shift']
    pm = row['Pm']
    cm = row['cm']
    cm_exact = float(EXPECTED_CM[N])
    sign_str = "+" if cm > 0 else "-"
    print(f"{N:>3}  {shift:>12.4f}  {pm:>10.4f}  {cm:>10.4f}  {str(EXPECTED_CM[N]):>12s}  {sign_str:>6}")
    # Verify against exact
    assert abs(cm - cm_exact) < 0.1, f"c_m mismatch at N={N}: got {cm:.4f}, expected {cm_exact}"

print()
print("Sign change at N=6: blob correction switches from DESTABILISING to STABILISING.")

## Stability consequences

The sign of $c_m$ has direct physical meaning for each $N$:

- **$N \le 5$: $c_m = -1$** -- The blob *destabilises* the already-stable polygon.
  The eigenvalue decreases: $\lambda_m(\varepsilon) < \lambda_m(0)$.
  Stability is maintained as long as $\varepsilon$ is small enough.

- **$N = 6$: $c_m = 13/12 > 0$** -- The blob *enhances* stability.
  The minimum eigenvalue increases with $\varepsilon$.

- **$N = 7$: $c_m = 4 > 0$** -- The marginal mode ($\lambda_m(0) = 0$) is
  *resolved*: $\lambda_m(\varepsilon) = 4\varepsilon^2 > 0$. The heptagon
  becomes strictly stable for any $\varepsilon > 0$.

- **$N = 8$: $c_m = 11 > 0$** -- The unstable mode ($\lambda_m(0) < 0$) is
  stabilised for $\varepsilon > \varepsilon_{\text{stab}}$.

In [ ]:
# Havelock eigenvalue: lambda_m(0) = (N-1) - m(N-m)/2, m = floor(N/2)
def havelock_eigenvalue(N):
    """Havelock eigenvalue for the critical mode m = floor(N/2)."""
    m = N // 2
    return (N - 1) - m * (N - m) / 2

print("Stability consequences of blob correction")
print("=" * 70)
print(f"{'N':>3}  {'lam_m(0)':>10}  {'c_m':>8}  {'effect':>40}")
print("-" * 70)

for N in range(3, 9):
    lam0 = havelock_eigenvalue(N)
    cm = table[N]['cm']

    if N <= 5:
        effect = f"destabilises (lam decreases by eps^2)"
    elif N == 6:
        effect = f"ENHANCES stability (c_m = 13/12 > 0)"
    elif N == 7:
        eps_stab = stabilization_threshold(N)
        effect = f"marginal RESOLVED: lam = 4*eps^2 > 0"
    else:
        eps_stab = stabilization_threshold(N)
        effect = f"stabilised for eps > {eps_stab:.4f}"

    print(f"{N:>3}  {lam0:>10.4f}  {cm:>8.4f}  {effect:>40}")

print()
print("N=8 stabilisation threshold:")
eps_8 = stabilization_threshold(8)
print(f"  eps_stab(8) = 1/sqrt(11) = {1/np.sqrt(11):.4f}")
print(f"  computed:     {eps_8:.4f}")
assert abs(eps_8 - 1/np.sqrt(11)) < 0.01, "N=8 threshold mismatch"

## Concentration condition delta(N)

For the blob correction to be a valid perturbation, the blob width $\varepsilon$ must
be small enough that the $O(\varepsilon^2)$ term dominates higher-order corrections.

The concentration condition from the operator norm bound gives:

$$
\delta(N) = \sqrt{\frac{\lambda_{\min}(N)}{6(2N-3)}}
$$

where $\lambda_{\min}(N)$ is the minimum Havelock eigenvalue (over all modes $m$).
For $\varepsilon < \delta(N) \cdot d_{\min}^2$, the blob correction does not
overcome the stability margin.

**Note**: For $N=7$, $\lambda_{\min} = 0$ (marginal mode), so the concentration
bound does not apply; the quartic normal form analysis is needed instead.

In [ ]:
def min_havelock_eigenvalue(N):
    """Minimum Havelock eigenvalue over all modes m = 1, ..., N-1."""
    evals = [(N - 1) - m * (N - m) / 2 for m in range(1, N)]
    return min(evals)


def concentration_bound(N):
    """delta(N) = sqrt(lambda_min / (6*(2N-3)))."""
    lam_min = min_havelock_eigenvalue(N)
    if lam_min <= 0:
        return None  # bound does not apply
    return np.sqrt(lam_min / (6 * (2 * N - 3)))


print("Concentration condition delta(N)")
print("=" * 55)
print(f"{'N':>3}  {'lam_min':>10}  {'6(2N-3)':>10}  {'delta(N)':>10}  {'note':>15}")
print("-" * 55)

for N in range(3, 9):
    lam_min = min_havelock_eigenvalue(N)
    coeff = 6 * (2 * N - 3)
    delta = concentration_bound(N)

    if delta is not None:
        note = ""
        print(f"{N:>3}  {lam_min:>10.4f}  {coeff:>10d}  {delta:>10.4f}  {note:>15}")
    else:
        note = "use quartic" if N == 7 else "unstable"
        print(f"{N:>3}  {lam_min:>10.4f}  {coeff:>10d}  {'---':>10}  {note:>15}")

print()
print("Verification against expected values:")
expected_delta = {4: 0.18, 5: 0.15, 6: 0.096}
for N, expected in expected_delta.items():
    delta = concentration_bound(N)
    print(f"  N={N}: delta = {delta:.3f}  (expected ~ {expected})")
    assert abs(delta - expected) < 0.02, f"delta({N}) mismatch"

## Operator norm bound

The operator norm of the blob Hessian correction satisfies:

$$
\|\nabla^2 H_\varepsilon - \nabla^2 H_0\|_{\mathrm{op}} \le \frac{6(2N-3)\,\varepsilon^2}{d_{\min}^4}
$$

where $d_{\min} = 2\sin(\pi/N)$ is the minimum inter-vortex distance on the
unit ring. We verify this bound numerically by comparing the actual spectral
norm of the blob Hessian correction against the analytic upper bound.

In [ ]:
from planetary_polygons.extensions.blob_correction import _analytic_blob_hessian_correction


def blob_hessian_eps(N, eps):
    """
    Numerical Hessian of blob energy at the N-gon for blob width eps.
    
    H_eps = -1/2 sum_{j<k} ln(|z_j - z_k|^2 + 2*eps^2)
    """
    z = np.exp(2j * np.pi * np.arange(N) / N)
    pos = np.concatenate([z.real, z.imag])
    dim = 2 * N

    def energy(p):
        x, y = p[:N], p[N:]
        H = 0.0
        for j in range(N):
            for k in range(j + 1, N):
                dx = x[j] - x[k]
                dy = y[j] - y[k]
                H -= 0.5 * np.log(dx**2 + dy**2 + 2 * eps**2)
        return H

    # Numerical Hessian via central differences
    h = 1e-5
    H_mat = np.zeros((dim, dim))
    for i in range(dim):
        for j in range(i, dim):
            p_pp = pos.copy(); p_pp[i] += h; p_pp[j] += h
            p_pm = pos.copy(); p_pm[i] += h; p_pm[j] -= h
            p_mp = pos.copy(); p_mp[i] -= h; p_mp[j] += h
            p_mm = pos.copy(); p_mm[i] -= h; p_mm[j] -= h
            val = (energy(p_pp) - energy(p_pm) - energy(p_mp) + energy(p_mm)) / (4 * h**2)
            H_mat[i, j] = val
            H_mat[j, i] = val
    return H_mat


def point_hessian(N):
    """Hessian of point-vortex energy at the N-gon (eps=0)."""
    return blob_hessian_eps(N, eps=0.0)


print("Operator norm bound verification")
print("=" * 70)
print(f"{'N':>3}  {'eps':>6}  {'d_min':>8}  {'||dH||_op':>12}  {'bound':>12}  {'ratio':>8}  {'OK?':>5}")
print("-" * 70)

test_cases = [(4, 0.05), (5, 0.05), (6, 0.03), (6, 0.05), (8, 0.05)]

for N, eps in test_cases:
    d_min = 2 * np.sin(np.pi / N)
    
    # Actual spectral norm of difference
    H_eps = blob_hessian_eps(N, eps)
    H_0 = point_hessian(N)
    diff = H_eps - H_0
    op_norm = np.max(np.abs(np.linalg.eigvalsh(diff)))
    
    # Analytic upper bound
    bound = 6 * (2 * N - 3) * eps**2 / d_min**4
    
    ratio = op_norm / bound
    ok = "YES" if op_norm <= bound * 1.01 else "NO"  # 1% tolerance for numerics
    print(f"{N:>3}  {eps:>6.3f}  {d_min:>8.4f}  {op_norm:>12.6f}  {bound:>12.6f}  {ratio:>8.4f}  {ok:>5}")

print()
print("The ratio ||dH||_op / bound should be < 1 (the bound is not tight).")

## Summary

**Key results verified:**

1. **$P_m$ values** match the paper table to within numerical tolerance ($<5\%$).

2. **Total correction $c_m$** reproduces the exact values:
   $\{-1,\, -1,\, -1,\, 13/12,\, 4,\, 11\}$ for $N = 3,\dots,8$.

3. **Sign change at $N=6$**: For $N \le 5$, the blob correction is uniformly
   destabilising ($c_m < 0$). For $N \ge 6$, the blob correction stabilises
   ($c_m > 0$).

4. **$N=7$ marginal mode resolution**: The exactly zero Havelock eigenvalue
   becomes $\lambda_7 = 4\varepsilon^2 > 0$ for any $\varepsilon > 0$.

5. **$N=8$ stabilisation threshold**: $\varepsilon_{\text{stab}} = 1/\sqrt{11} \approx 0.30$.

6. **Concentration bound** $\delta(N)$: $N=4: 0.18$, $N=5: 0.15$, $N=6: 0.096$.
   For $N=7$ the bound is vacuous (marginal mode); the quartic normal form
   analysis handles that case.

7. **Operator norm bound**: $\|\nabla^2 H_\varepsilon - \nabla^2 H_0\|_{\text{op}} \le 6(2N-3)\varepsilon^2/d_{\min}^4$
   is satisfied in all tested cases.

These results confirm Proposition 8 of the paper: the Cauchy blob regularisation
provides a controlled, profile-independent mechanism that enhances stability
for $N \ge 6$ and resolves the $N=7$ marginal mode.